In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
NOTEBOOK_NAME = "opt_baselines_and_simmim"    # identity + build fingerprint of THESE cells;
NOTEBOOK_BUILD = "c841b8f849d2"  # checked against the repo so stale cells fail loudly
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/christopherh-88/HierarchicalDet.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

# `git pull` above refreshed src/ and configs_repro/ -- but NOT these cells,
# which are the copy uploaded to Kaggle. Fail loudly rather than run a mix.
print("notebook build:", setup_env.assert_notebook_current(NOTEBOOK_NAME, NOTEBOOK_BUILD))
setup_env.install_dependencies()
# The vendored pycocotools ships Python sources only; its compiled `_mask`
# extension is grafted in here and VERIFIED BY IMPORT. It is compiled against
# numpy's C ABI, so a mismatch surfaces as "numpy.dtype size changed" deep
# inside detectron2.structures — which reads as a detectron2 problem and is not.
import numpy
print("numpy {} | pycocotools _mask -> {}".format(
    numpy.__version__, setup_env.ensure_pycocotools_mask()))
run = setup_env.bootstrap(RUN_MODE, require_gpu=True)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- Opt-in flags ----
from src import data_convert

RUN_BASELINES = False
RUN_SIMMIM = False
BASELINE_MODELS = ("retinanet", "faster_rcnn")
BASELINE_TIERS = (2,)
BASELINE_MAX_ITER = 20000 if run.mode == "full" else 10000
SIMMIM_EPOCHS = 100

paths = data_convert.layout()
print("baselines:", RUN_BASELINES, "| SimMIM:", RUN_SIMMIM)


In [ ]:
# ---- Baselines, via the existing tools/baselines driver ----
baseline_records = []
if RUN_BASELINES:
    for model in BASELINE_MODELS:
        for tier in BASELINE_TIERS:
            output = os.path.join(setup_env.RUNS_DIR,
                                  "baseline_{}_tier{}".format(model, tier))
            command = [sys.executable, os.path.join("tools", "baselines", "train_baseline.py"),
                       "--model", model,
                       "--train-json", data_convert.flat_json_path(tier, "train"),
                       "--train-images", paths["img_diagnosis"],
                       "--test-json", data_convert.flat_json_path(tier, "test"),
                       "--test-images", paths["img_test"],
                       "--output-dir", output, "--tier", "0",
                       "--max-iter", str(BASELINE_MAX_ITER),
                       "--seed", str(setup_env.BASE_SEED)]
            print("$", " ".join(command))
            result = subprocess.run(command, cwd=setup_env.REPO_ROOT)
            baseline_records.append({"model": model, "tier": tier, "output_dir": output,
                                     "returncode": result.returncode,
                                     "command": " ".join(command)})
            if result.returncode != 0:
                raise RuntimeError("baseline {} tier {} failed".format(model, tier))
    setup_env.log_deviation(
        "baselines are our own training runs, not the authors' configurations",
        "the repository ships no RetinaNet / Faster R-CNN / DETR config, so these "
        "reproduce the comparison rather than the authors' specific baseline runs",
        "opt_baselines_and_simmim")
print(json.dumps(baseline_records, indent=2))


In [ ]:
# ---- SimMIM pretraining on the 1,571 unlabelled X-rays ----
simmim = {"ran": False}
if RUN_SIMMIM:
    unlabelled = paths["img_unlabelled"]
    if not (os.path.isdir(unlabelled) and os.listdir(unlabelled)):
        data_convert.download_and_extract(
            os.path.join(setup_env.PROJECT_ROOT, "dentex_raw"),
            include_unlabelled=True, delete_archives=True)
    print("unlabelled images:", len(os.listdir(unlabelled)))

    clone = os.path.join(setup_env.PROJECT_ROOT, "Swin-Transformer")
    if not os.path.isdir(os.path.join(clone, ".git")):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/microsoft/Swin-Transformer.git", clone],
                       check=True)
    output = os.path.join(setup_env.RUNS_DIR, "simmim_pretrain")
    os.makedirs(output, exist_ok=True)
    command = [sys.executable, os.path.join(clone, "main_simmim.py"),
               "--cfg", os.path.join(clone, "configs", "simmim",
                                     "simmim_pretrain__swin_base__img192_window6__100ep.yaml"),
               "--data-path", os.path.dirname(unlabelled),
               "--batch-size", "16", "--output", output,
               "--opts", "TRAIN.EPOCHS", str(SIMMIM_EPOCHS)]
    print("$", " ".join(command))
    result = subprocess.run(command, cwd=clone)
    simmim = {"ran": True, "returncode": result.returncode, "output_dir": output,
              "command": " ".join(command), "epochs": SIMMIM_EPOCHS}
    if result.returncode == 0:
        setup_env.log_deviation(
            "SimMIM pretraining WAS performed in this run",
            "the optional notebook was executed, so backbone initialization follows the "
            "paper's description rather than the released nonpretrain config",
            "opt_baselines_and_simmim")
print(json.dumps(simmim, indent=2))


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {"run_mode": run.mode, "baselines": baseline_records, "simmim": simmim}

path = setup_env.write_notebook_summary("opt_baselines_and_simmim", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])
